# 02 — Download dos Dados ABIDE

Este notebook realiza o download e a organização dos dados do
ABIDE utilizados no estudo de classificação de Transtorno do
Espectro Autista (TEA).

O processamento e a análise dos dados serão realizados em notebooks
posteriores.

**Objetivo:** Baixar o ABIDE I (Preprocessed Connectomes Project, CC200) e organizar fenótipos e séries temporais.

**Entradas:** Internet (somente se os dados ainda não existirem em `Data/`).

**Processamento:** `fetch_abide_pcp` (cpac, band-pass, sem GSR, `rois_cc200`, quality_checked). O download é pulado se os artefatos finais já existirem.

**Saídas:** `Data/raw/ABIDE_pcp/`, `Data/raw/phenotypic_abide.csv`, `Data/roi_timeseries/roi_timeseries_abide.pkl`.

In [10]:
# ------------------------------------------------------------
# Configuração do projeto (execução local — VS Code)
# Garante que a raiz do projeto esteja no sys.path, funcionando tanto se o
# diretório de trabalho do kernel for notebooks/ quanto a raiz do projeto.
# ------------------------------------------------------------
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

Projeto (BASE_DIR) = C:\Users\Maria Eduarda\Downloads\TCC_TEA_TDAH


In [11]:
import pickle
import pandas as pd
import nilearn
import nibabel as nib

print("Nilearn:", nilearn.__version__)
print("NiBabel:", nib.__version__)

from nilearn.datasets import fetch_abide_pcp

Nilearn: 0.14.1
NiBabel: 5.4.2


Download ABIDE I

In [12]:
from nilearn.datasets import fetch_abide_pcp

# Verificação condicional: se os artefatos finais já existem, não baixa de novo.
_csv_fenotipico = PATHS['raw'] / 'phenotypic_abide.csv'
_pkl_series = PATHS['roi_timeseries'] / 'roi_timeseries_abide.pkl'

if _csv_fenotipico.exists() and _pkl_series.exists():
    abide = None
    print('Dados já existem — download ignorado:')
    print(' ', _csv_fenotipico)
    print(' ', _pkl_series)
else:
    abide = fetch_abide_pcp(
        data_dir=str(PATHS['raw']),
        n_subjects=None,
        pipeline="cpac",
        band_pass_filtering=True,
        global_signal_regression=False,
        derivatives=["rois_cc200"],
        quality_checked=True
    )

    print("Download concluído!")
    print("Total de sujeitos:", len(abide.phenotypic))

Dados já existem — download ignorado:
  C:\Users\Maria Eduarda\Downloads\TCC_TEA_TDAH\Data\raw\phenotypic_abide.csv
  C:\Users\Maria Eduarda\Downloads\TCC_TEA_TDAH\Data\roi_timeseries\roi_timeseries_abide.pkl


In [14]:
# ============================================================
# METADADOS FENOTÍPICOS
# ============================================================
#
# DX_GROUP:
# 1 = TEA
# 2 = Controle
#
# SUB_ID será tratado como string para garantir
# correspondência consistente entre fenótipo e séries temporais.
# ============================================================

if abide is None:

    phenotypic = pd.read_csv(
        PATHS['raw'] / 'phenotypic_abide.csv'
    )

    print(
        'phenotypic_abide.csv já existe — usando o arquivo existente.'
    )

else:

    phenotypic = pd.DataFrame(
        abide.phenotypic
    )

    phenotypic.to_csv(
        PATHS['raw'] / 'phenotypic_abide.csv',
        index=False
    )

# ------------------------------------------------------------
# Padronizar SUB_ID como string
# ------------------------------------------------------------

phenotypic['SUB_ID'] = (
    phenotypic['SUB_ID']
    .astype(str)
)

# ------------------------------------------------------------
# Conferência
# ------------------------------------------------------------

print(
    phenotypic[
        [
            'SUB_ID',
            'SITE_ID',
            'DX_GROUP',
            'AGE_AT_SCAN',
            'SEX'
        ]
    ].head()
)

print()
print(
    'Quantidade de participantes:',
    len(phenotypic)
)

print()
print(
    'Distribuição por grupo diagnóstico '
    '(1=TEA, 2=Controle):'
)

print(
    phenotypic['DX_GROUP'].value_counts()
)

phenotypic_abide.csv já existe — usando o arquivo existente.
  SUB_ID SITE_ID  DX_GROUP  AGE_AT_SCAN  SEX
0  50003    PITT         1        24.45    1
1  50004    PITT         1        19.09    1
2  50005    PITT         1        13.73    2
3  50006    PITT         1        13.37    1
4  50007    PITT         1        17.78    1

Quantidade de participantes: 871

Distribuição por grupo diagnóstico (1=TEA, 2=Controle):
DX_GROUP
2    468
1    403
Name: count, dtype: int64


In [15]:
# ============================================================
# SALVAR / CARREGAR SÉRIES TEMPORAIS DAS ROIs
# ============================================================
#
# Cada participante possui uma matriz:
#
#     tempo × 200 ROIs
#
# O número de pontos temporais pode variar entre participantes.
# ============================================================

import pickle

out_path = (
    PATHS['roi_timeseries']
    / 'roi_timeseries_abide.pkl'
)

# ------------------------------------------------------------
# Caso o download tenha sido realizado nesta execução
# ------------------------------------------------------------

if abide is not None:

    roi_timeseries = {}

    for sub_id, ts in zip(
        phenotypic['SUB_ID'],
        abide.rois_cc200
    ):

        roi_timeseries[str(sub_id)] = np.asarray(ts)

    with open(
        out_path,
        'wb'
    ) as f:

        pickle.dump(
            roi_timeseries,
            f
        )

    print(
        f'{len(roi_timeseries)} séries temporais salvas em:'
    )

    print(out_path)

# ------------------------------------------------------------
# Caso os dados já existam
# ------------------------------------------------------------

else:

    if not out_path.exists():

        raise FileNotFoundError(
            f'Arquivo de séries temporais não encontrado:\n'
            f'{out_path}'
        )

    with open(
        out_path,
        'rb'
    ) as f:

        roi_timeseries = pickle.load(f)

    print(
        'Séries temporais já existentes — arquivo carregado.'
    )

    print(
        'Arquivo:',
        out_path
    )

# ------------------------------------------------------------
# Conferência
# ------------------------------------------------------------

print()
print(
    'Número de sujeitos:',
    len(roi_timeseries)
)

print(
    'Exemplo de SUB_ID:',
    next(iter(roi_timeseries.keys()))
)

print(
    'Shape de uma série:',
    next(iter(roi_timeseries.values())).shape
)

Séries temporais já existentes — arquivo carregado.
Arquivo: C:\Users\Maria Eduarda\Downloads\TCC_TEA_TDAH\Data\roi_timeseries\roi_timeseries_abide.pkl

Número de sujeitos: 871
Exemplo de SUB_ID: 50003
Shape de uma série: (196, 200)


## Dimensão das séries temporais

As séries temporais extraídas pelo pipeline CPAC utilizando o
atlas CC200 possuem **200 ROIs** por participante.

O número de volumes temporais pode variar entre os participantes,
em função das características das aquisições e do pré-processamento
dos diferentes estudos que compõem o ABIDE I.

Portanto, cada participante é representado por uma matriz:

**tempo × 200 ROIs**

com número de pontos temporais potencialmente diferente entre os
participantes.

Essa diferença não impede o cálculo da conectividade funcional,
pois a correlação de Pearson é calculada entre as séries temporais
das ROIs de cada participante.

In [16]:
# ============================================================
# AUDITORIA DAS SÉRIES TEMPORAIS
# ============================================================

from collections import Counter

print('==========================================')
print('AUDITORIA DAS SÉRIES TEMPORAIS')
print('==========================================')

# ------------------------------------------------------------
# Quantidade de sujeitos
# ------------------------------------------------------------

print(
    'Número de sujeitos:',
    len(roi_timeseries)
)

# ------------------------------------------------------------
# Distribuição dos shapes
# ------------------------------------------------------------

shapes = Counter(
    tuple(np.asarray(ts).shape)
    for ts in roi_timeseries.values()
)

print()
print('Shapes encontrados:')

for shape, quantidade in sorted(
    shapes.items()
):

    print(
        f'  {shape}: {quantidade} sujeitos'
    )

# ------------------------------------------------------------
# Verificar número de ROIs
# ------------------------------------------------------------

n_rois = set(
    np.asarray(ts).shape[1]
    for ts in roi_timeseries.values()
)

print()
print(
    'Número de ROIs encontrados:',
    n_rois
)

if n_rois != {200}:

    raise ValueError(
        'ERRO: nem todos os participantes possuem 200 ROIs.'
    )

# ------------------------------------------------------------
# Verificar NaN / infinito
# ------------------------------------------------------------

problemas = []

for sub_id, ts in roi_timeseries.items():

    ts = np.asarray(ts)

    if not np.isfinite(ts).all():

        problemas.append(sub_id)

print()
print(
    'Sujeitos com NaN/Inf:',
    len(problemas)
)

if problemas:

    print(
        'Exemplos:',
        problemas[:10]
    )

# ------------------------------------------------------------
# Verificar correspondência com fenótipo
# ------------------------------------------------------------

ids_series = set(
    str(x)
    for x in roi_timeseries.keys()
)

ids_phenotypic = set(
    phenotypic['SUB_ID']
    .astype(str)
)

sem_phenotypic = (
    ids_series -
    ids_phenotypic
)

sem_series = (
    ids_phenotypic -
    ids_series
)

print()
print(
    'Séries sem registro fenotípico:',
    len(sem_phenotypic)
)

print(
    'Participantes sem série temporal:',
    len(sem_series)
)

# ------------------------------------------------------------
# Resultado final
# ------------------------------------------------------------

if (
    len(sem_phenotypic) == 0
    and len(sem_series) == 0
    and n_rois == {200}
    and len(problemas) == 0
):

    print()
    print('==========================================')
    print('AUDITORIA CONCLUÍDA COM SUCESSO')
    print('==========================================')

else:

    print()
    print('==========================================')
    print('ATENÇÃO: existem inconsistências')
    print('==========================================')

AUDITORIA DAS SÉRIES TEMPORAIS
Número de sujeitos: 871

Shapes encontrados:
  (78, 200): 25 sujeitos
  (116, 200): 119 sujeitos
  (124, 200): 4 sujeitos
  (146, 200): 59 sujeitos
  (152, 200): 29 sujeitos
  (176, 200): 211 sujeitos
  (196, 200): 129 sujeitos
  (202, 200): 1 sujeitos
  (206, 200): 28 sujeitos
  (232, 200): 1 sujeitos
  (236, 200): 86 sujeitos
  (246, 200): 56 sujeitos
  (296, 200): 120 sujeitos
  (316, 200): 3 sujeitos

Número de ROIs encontrados: {200}

Sujeitos com NaN/Inf: 0

Séries sem registro fenotípico: 0
Participantes sem série temporal: 0

AUDITORIA CONCLUÍDA COM SUCESSO
